In [1]:
import os
import pandas as pd
import ifcopenshell
import ifcopenshell.util.element
import ifcopenshell.api
import joblib
import json
import numpy as np
import traceback
from sklearn.feature_extraction.text import TfidfVectorizer # Importar aqui
from sklearn.utils.class_weight import compute_sample_weight # Necessário se for usar pesos no futuro

In [2]:
# --- 1. Pastas de Entrada e Saída ---
INPUT_FOLDER_PATH = r"C:\Users\lucas.galicioli\Downloads\ifc-teste-24_10 - Entradas"  # <-- IMPORTANTE: Defina a pasta com os IFCs a processar
OUTPUT_FOLDER_PATH = r"C:\Users\lucas.galicioli\Downloads\ifc-teste-24_10 - Resultados" # <-- IMPORTANTE: Defina a pasta onde os resultados serão salvos

# --- 2. Caminhos dos Artefatos ---
# Pasta onde os modelos por disciplina (com TFIDF) foram salvos
MODEL_ARTIFACTS_DIR = "modelos_por_disciplina_tfidf" 
# Caminho para a matriz de classificação (para mapear disciplina)
MATRIX_PATH = r"C:\Users\lucas.galicioli\Desktop\ifc-classifier\data\interim\classification-matrix.xlsx" # Use seu caminho
# Colunas usadas no mapeamento de disciplina
COLUNA_MATRIX_FILENAME = 'FileName'            
COLUNA_MATRIX_DISCIPLINA = 'Ô_CLS_DISCIPLINAS' 
# Arquivo de medianas (carregado mas não usado ativamente se o modelo não usa dimensões)
MEDIANS_PATH = 'medianas_treinamento.json'

# --- 3. Configurações de Classificação e Saída ---
PSET_NAME = "SOLIBRI_CLASSIFICACAO" 
# Limiar de confiança para aceitar a previsão do modelo (Ex: 0.80 = 80%)
# Ajuste conforme necessário após testes
CONFIDENCE_THRESHOLD = 0.60 
# Nome exato da classe "Unclassified" usada no treino
UNCLASSIFIED_TEXT = "Unclassified" 

# --- Criação da Pasta de Saída ---
os.makedirs(OUTPUT_FOLDER_PATH, exist_ok=True)
print(f"Pasta de entrada: {INPUT_FOLDER_PATH}")
print(f"Pasta de saída: {OUTPUT_FOLDER_PATH}")
print(f"Pasta de artefatos: {MODEL_ARTIFACTS_DIR}")

Pasta de entrada: C:\Users\lucas.galicioli\Downloads\ifc-teste-24_10 - Entradas
Pasta de saída: C:\Users\lucas.galicioli\Downloads\ifc-teste-24_10 - Resultados
Pasta de artefatos: modelos_por_disciplina_tfidf


In [3]:
def get_building_storey(element):
    """ Encontra o IfcBuildingStorey no qual o elemento está contido. """
    try:
        spatial_container = ifcopenshell.util.element.get_container(element)
        if spatial_container and spatial_container.is_a('IfcBuildingStorey'):
            return spatial_container.Name
    except Exception:
        pass
    return None

def get_material_name(element):
    """ Extrai o nome do material associado ao elemento. """
    material = ifcopenshell.util.element.get_material(element)
    if not material: return None
    if hasattr(material, 'Name'): return material.Name
    elif hasattr(material, 'MaterialLayers'):
        layer_names = [
            layer.Material.Name for layer in material.MaterialLayers
            if hasattr(layer, 'Material') and hasattr(layer.Material, 'Name')
        ]
        return ', '.join(layer_names) if layer_names else None
    return None

def get_quantity_value_legacy(element, quantity_name):
    """ Busca por uma quantidade específica (ex: 'Width') e retorna seu valor. """
    for definition in getattr(element, 'IsDefinedBy', []):
        if definition.is_a('IfcRelDefinesByProperties'):
            prop_set = definition.RelatingPropertyDefinition
            if prop_set.is_a('IfcElementQuantity'):
                for quantity in prop_set.Quantities:
                    if quantity.Name == quantity_name:
                        value_attribute = next((attr for attr in dir(quantity) if attr.endswith('Value')), None)
                        if value_attribute:
                            return getattr(quantity, value_attribute)
    return None

def gerar_mapa_disciplinas(matrix_path, col_filename, col_disciplina):
    """
    Carrega a matriz de classificação e cria um dicionário de mapeamento
    (Código -> Disciplina). Ex: {'HID': 'Hidrossanitário', 'EST': 'Estrutura'}
    """
    try:
        df_matrix = pd.read_excel(matrix_path)
        df_mapa_temp = df_matrix[[col_filename, col_disciplina]].copy()
        df_mapa_temp['Disciplina_Code'] = df_mapa_temp[col_filename].str.split('-').str[1]
        df_mapa_temp = df_mapa_temp[['Disciplina_Code', col_disciplina]].dropna().drop_duplicates()
        mapa = df_mapa_temp.set_index('Disciplina_Code')[col_disciplina].to_dict()
        
        if not mapa:
            print(f"AVISO: O mapa de disciplinas gerado a partir de '{matrix_path}' está vazio.")
        return mapa
    except FileNotFoundError:
        print(f"ERRO: Arquivo da matriz não encontrado em: {matrix_path}")
    except KeyError as e:
        print(f"ERRO: Coluna {e} não encontrada na matriz. Verifique os nomes '{col_filename}' e '{col_disciplina}'.")
    except Exception as e:
        print(f"ERRO ao gerar mapa de disciplinas: {e}")
    return None

def extrair_disciplina_do_nome(nome_arquivo_base, mapa_disciplinas):
    """ Extrai o código do nome do arquivo e o traduz usando o mapa. """
    try:
        nome_base = os.path.splitext(nome_arquivo_base)[0]
        partes_nome = nome_base.split('-')
        codigo_disciplina = partes_nome[1] # Pega o "HID", "EST", etc.
        
        disciplina_traduzida = mapa_disciplinas.get(codigo_disciplina, f"Código '{codigo_disciplina}' Não Mapeado")
        return disciplina_traduzida
    except IndexError:
        print(f"AVISO: O nome '{nome_arquivo_base}' não segue o padrão 'XXX-CODIGO-...'.")
        return "Erro: Padrão de Nome"
    except Exception:
        return "Erro: Extração"

In [4]:
# Função que processa UM ÚNICO arquivo IFC
def processar_single_ifc(input_ifc_path, output_folder_path, mapa_disciplinas, 
                         model_artifacts_dir, pset_name, confidence_threshold, 
                         unclassified_text, medianas_treinamento):
    
    file_name = os.path.basename(input_ifc_path)
    print(f"\n--- Processando Arquivo: {file_name} ---")
    
    try:
        # --- ETAPA 1: Extrair dados do IFC ---
        print("   Extraindo dados...")
        ifc_file = ifcopenshell.open(input_ifc_path)
        products = ifc_file.by_type('IfcProduct')
        element_data = []
        for product in products:
             # (Loop de extração igual ao anterior)
            if product.is_a('IfcOpeningElement') or product.is_a('IfcVirtualElement'): continue
            psets = ifcopenshell.util.element.get_psets(product)
            rogga_pset = psets.get('PSET_RÔGGA', {})
            element_info = {
                 'GlobalId': product.GlobalId, 'Class': product.is_a(),
                 'PredefinedType': getattr(product, 'PredefinedType', None),
                 'Name': getattr(product, 'Name', None), 
                 'BuildingStorey': get_building_storey(product),
                 'Material': get_material_name(product),
                 'PSET_RÔGGA.RÔGGA_SEÇÃO': rogga_pset.get('RÔGGA_SEÇÃO', None),
                 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO': rogga_pset.get('RÔGGA_DESCRIÇÃO', None),
                 'Width': get_quantity_value_legacy(product, 'Width'), # Manter extração caso precise no futuro
                 'Thickness': get_quantity_value_legacy(product, 'Thickness'),
                 'Length': get_quantity_value_legacy(product, 'Length'),
                 'Height': get_quantity_value_legacy(product, 'Height'),
                 'FileName': file_name # Adicionar o nome do arquivo aqui pode ser útil
            }
            element_data.append(element_info)
            
        df_ifc_data = pd.DataFrame(element_data)
        if df_ifc_data.empty:
            print(f"   AVISO: Nenhum elemento válido extraído de {file_name}. Pulando.")
            return False # Indicar falha

        # --- ETAPA 2: Determinar Disciplina ---
        print("   Determinando disciplina...")
        disciplina_arquivo = extrair_disciplina_do_nome(file_name, mapa_disciplinas)
        df_ifc_data['Ô_CLS_DISCIPLINAS'] = disciplina_arquivo
        print(f"   -> Disciplina: '{disciplina_arquivo}'")
        if "Erro" in disciplina_arquivo or "Não Mapeado" in disciplina_arquivo:
             print(f"   ERRO: Disciplina inválida para {file_name}. Pulando.")
             return False

        # --- ETAPA 3: Carregar Artefatos Específicos ---
        print("   Carregando artefatos da disciplina...")
        discipline_code = str(disciplina_arquivo).replace(' ', '_').replace('/', '-')
        MODEL_PATH_DISC = os.path.join(model_artifacts_dir, f"modelo_{discipline_code}.pkl")
        ENCODER_PATH_DISC = os.path.join(model_artifacts_dir, f"encoder_le1_{discipline_code}.pkl")
        COLUMNS_PATH_DISC = os.path.join(model_artifacts_dir, f"colunas_{discipline_code}.json")
        TFIDF_PATH_DISC = os.path.join(model_artifacts_dir, f"tfidf_{discipline_code}.pkl")
        try:
            modelo = joblib.load(MODEL_PATH_DISC)
            label_encoder = joblib.load(ENCODER_PATH_DISC)
            with open(COLUMNS_PATH_DISC, 'r', encoding='utf-8') as f:
                colunas_do_modelo = json.load(f)
            tfidf_vectorizer = joblib.load(TFIDF_PATH_DISC)
        except FileNotFoundError:
            print(f"   ERRO: Artefatos não encontrados para disciplina '{disciplina_arquivo}' em '{model_artifacts_dir}'. Pulando arquivo.")
            return False

        # --- ETAPA 4: Pré-processamento e Classificação ---
        print("   Iniciando pré-processamento...")
        # (Lógica de pré-processamento com TFIDF, FE, OHE, Reindex - Mantida igual)
        features_iniciais = [
             'Class', 'PredefinedType', 'BuildingStorey', 
             'Material', 'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO', 'Name'
        ]
        features_presentes = [col for col in features_iniciais if col in df_ifc_data.columns]
        df_para_prever = df_ifc_data[features_presentes].copy()
        placeholder = "Desconhecido" 
        cat_cols_all = df_para_prever.select_dtypes(include=['object']).columns
        df_para_prever.loc[:, cat_cols_all] = df_para_prever.loc[:, cat_cols_all].fillna(placeholder)
        
        # TF-IDF Transform
        tfidf_features_app = tfidf_vectorizer.transform(df_para_prever['Name'])
        tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
        df_tfidf_app = pd.DataFrame(tfidf_features_app.toarray(), columns=tfidf_feature_names, index=df_para_prever.index)
        df_para_prever = df_para_prever.drop('Name', axis=1)
        df_para_prever = pd.concat([df_para_prever, df_tfidf_app], axis=1)

        # Frequency Encoding
        colunas_para_freq_encoding = ['Material', 'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO']
        for col in colunas_para_freq_encoding:
             if col in df_para_prever.columns:
                 frequencias = df_para_prever[col].value_counts(normalize=True) 
                 df_para_prever[col + '_Freq'] = df_para_prever[col].map(frequencias).fillna(0)
                 df_para_prever = df_para_prever.drop(col, axis=1)
        
        # One-Hot Encoding
        df_encodado = pd.get_dummies(df_para_prever) 
        df_encodado.columns = df_encodado.columns.str.replace(r'[\[\]<]', '_', regex=True)

        # Reindex
        df_final = df_encodado.reindex(columns=colunas_do_modelo, fill_value=0)
        print(f"   -> Pré-processamento concluído. Shape: {df_final.shape}")

        # Classificação com Limiar
        print("   Realizando previsões com limiar...")
        probabilities = modelo.predict_proba(df_final)
        default_predictions_numeric = modelo.predict(df_final)
        max_probabilities = np.max(probabilities, axis=1)
        
        try:
            unclassified_label_numeric = np.where(label_encoder.classes_ == unclassified_text)[0][0]
        except IndexError:
             print(f"   AVISO: Classe '{unclassified_text}' não encontrada no encoder de '{disciplina_arquivo}'. Usando a primeira classe como fallback.")
             unclassified_label_numeric = 0
             
        final_predictions_numeric = np.full(len(df_final), unclassified_label_numeric, dtype=int)
        is_confident = max_probabilities >= confidence_threshold
        final_predictions_numeric[is_confident] = default_predictions_numeric[is_confident]
        num_unclassified = np.sum(~is_confident)
        print(f"   -> {num_unclassified / len(df_final):.1%} ficaram como '{unclassified_text}'.")
        
        previsoes_texto_solibri = label_encoder.inverse_transform(final_predictions_numeric)
        df_ifc_data['Ô_CLS_CLASSIFICAÇÃO_SOLIBRI'] = previsoes_texto_solibri
        
        # --- ETAPA 5: Escrever no IFC ---
        print("   Gravando propriedades no IFC...")
        df_mapa_final = df_ifc_data[['GlobalId', 'Ô_CLS_CLASSIFICAÇÃO_SOLIBRI', 'Ô_CLS_DISCIPLINAS']]
        mapa_de_escrita = df_mapa_final.set_index('GlobalId').to_dict('index')
        elementos_modificados = 0
        for global_id, propriedades in mapa_de_escrita.items():
             elemento = ifc_file.by_guid(global_id)
             if not elemento: continue
             try:
                # Lógica robusta para adicionar/editar Pset (igual à anterior)
                 existing_psets = ifcopenshell.util.element.get_psets(elemento)
                 pset = None
                 if pset_name not in existing_psets:
                     pset = ifcopenshell.api.run("pset.add_pset", ifc_file, product=elemento, name=pset_name)
                 else:
                     pset_info = existing_psets[pset_name] 
                     if isinstance(pset_info, dict) and 'id' in pset_info: pset = ifc_file.by_id(pset_info['id']) 
                     elif isinstance(pset_info, ifcopenshell.entity_instance): pset = pset_info
                     else: # Tentar encontrar pelo nome se get_psets falhou
                          for relDef in elemento.IsDefinedBy:
                              if relDef.is_a('IfcRelDefinesByProperties'):
                                  propSetDef = relDef.RelatingPropertyDefinition
                                  if propSetDef.is_a('IfcPropertySet') and propSetDef.Name == pset_name:
                                      pset = propSetDef; break
                          if not pset: pset = ifcopenshell.api.run("pset.add_pset", ifc_file, product=elemento, name=pset_name)
                 
                 if pset and isinstance(pset, ifcopenshell.entity_instance):
                      ifcopenshell.api.run("pset.edit_pset", ifc_file, pset=pset, properties=propriedades)
                      elementos_modificados += 1
                 else: print(f"   ERRO: Falha ao obter/criar Pset '{pset_name}' para {global_id}")
             except Exception as e:
                 print(f"   ERRO ao gravar Pset em {global_id}: {e}")
        print(f"   -> Propriedades gravadas em {elementos_modificados} elementos.")

        # --- ETAPA 6: Salvar Arquivos ---
        # Definir nomes de saída dinamicamente
        output_ifc_filename = f"classificado_{file_name}"
        output_ifc_path = os.path.join(output_folder_path, output_ifc_filename)
        
        print(f"   Salvando IFC modificado em: {output_ifc_path}")
        ifc_file.write(output_ifc_path)
       
        return True # Indicar sucesso

    except FileNotFoundError as e:
        print(f"   ERRO CRÍTICO ao abrir {file_name}: {e}")
        return False
    except Exception as e:
        print(f"   ERRO INESPERADO durante processamento de {file_name}: {e}")
        traceback.print_exc()
        return False

print("Função de processamento individual definida.")

Função de processamento individual definida.


In [5]:
# --- Loop Principal para Processar a Pasta ---

print("\n" + "="*50)
print("INICIANDO PROCESSAMENTO EM LOTE")
print("="*50)

# 1. Carregar Mapa de Disciplinas (feito uma vez)
print("Carregando mapa de disciplinas geral...")
mapa_disciplinas_global = gerar_mapa_disciplinas(MATRIX_PATH, COLUNA_MATRIX_FILENAME, COLUNA_MATRIX_DISCIPLINA)

# 2. Carregar Medianas (feito uma vez)
print("Carregando medianas gerais...")
try:
    with open(MEDIANS_PATH, 'r', encoding='utf-8') as f:
        medianas_global = json.load(f)
except FileNotFoundError:
    print(f"AVISO: Arquivo de medianas '{MEDIANS_PATH}' não encontrado.")
    medianas_global = {}

if not mapa_disciplinas_global:
    print("ERRO CRÍTICO: Não foi possível carregar o mapa de disciplinas. Abortando.")
else:
    # 3. Iterar pelos arquivos na pasta de entrada
    arquivos_processados = 0
    arquivos_com_erro = 0
    
    print(f"\nProcurando arquivos .ifc em: {INPUT_FOLDER_PATH}")
    for filename in os.listdir(INPUT_FOLDER_PATH):
        if filename.lower().endswith(".ifc"):
            full_input_path = os.path.join(INPUT_FOLDER_PATH, filename)
            
            # Chamar a função para processar este arquivo
            success = processar_single_ifc(
                input_ifc_path=full_input_path,
                output_folder_path=OUTPUT_FOLDER_PATH,
                mapa_disciplinas=mapa_disciplinas_global,
                model_artifacts_dir=MODEL_ARTIFACTS_DIR,
                pset_name=PSET_NAME,
                confidence_threshold=CONFIDENCE_THRESHOLD,
                unclassified_text=UNCLASSIFIED_TEXT,
                medianas_treinamento=medianas_global # Passa as medianas
            )
            
            if success:
                arquivos_processados += 1
            else:
                arquivos_com_erro += 1
        else:
            print(f"Ignorando arquivo não-IFC: {filename}")

    print("\n" + "="*50)
    print("PROCESSAMENTO EM LOTE CONCLUÍDO")
    print("="*50)
    print(f"Arquivos IFC processados com sucesso: {arquivos_processados}")
    print(f"Arquivos IFC com erro: {arquivos_com_erro}")
    print(f"Resultados salvos em: {OUTPUT_FOLDER_PATH}")


INICIANDO PROCESSAMENTO EM LOTE
Carregando mapa de disciplinas geral...
Carregando medianas gerais...

Procurando arquivos .ifc em: C:\Users\lucas.galicioli\Downloads\ifc-teste-24_10 - Entradas

--- Processando Arquivo: PHN21043-ARQ-AP-0001-BIM-EMB-GER-MODELO_IFC_EMBASAMENTO-R06.ifc ---
   Extraindo dados...
   Determinando disciplina...
   -> Disciplina: 'Arquitetura'
   Carregando artefatos da disciplina...
   Iniciando pré-processamento...
   -> Pré-processamento concluído. Shape: (14069, 162)
   Realizando previsões com limiar...


c:\Users\lucas.galicioli\Desktop\ifc-classifier\.venv\Lib\site-packages\xgboost\core.py:774: UserWarning: [16:44:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


   -> 7.3% ficaram como 'Unclassified'.
   Gravando propriedades no IFC...
   -> Propriedades gravadas em 14069 elementos.
   Salvando IFC modificado em: C:\Users\lucas.galicioli\Downloads\ifc-teste-24_10 - Resultados\classificado_PHN21043-ARQ-AP-0001-BIM-EMB-GER-MODELO_IFC_EMBASAMENTO-R06.ifc

--- Processando Arquivo: PHN21043-ARQ-AP-0002-BIM-TOR-GER-MODELO_IFC_TORRE-R06.ifc ---
   Extraindo dados...
   Determinando disciplina...
   -> Disciplina: 'Arquitetura'
   Carregando artefatos da disciplina...
   Iniciando pré-processamento...
   -> Pré-processamento concluído. Shape: (8792, 162)
   Realizando previsões com limiar...
   -> 11.4% ficaram como 'Unclassified'.
   Gravando propriedades no IFC...
   -> Propriedades gravadas em 8792 elementos.
   Salvando IFC modificado em: C:\Users\lucas.galicioli\Downloads\ifc-teste-24_10 - Resultados\classificado_PHN21043-ARQ-AP-0002-BIM-TOR-GER-MODELO_IFC_TORRE-R06.ifc

--- Processando Arquivo: PHN21043-ARQ-EX-0001-BIM-EMB-GER-R02.ifc ---
   Ext